# TP5 — Partie A : Agentic AI avec LangChain

Un agent = **MIND** (le LLM qui raisonne) + **BODY** (les tools qui agissent), piloté par le pattern **ReAct** :
*Raisonnement → Action (appel d'un tool) → Observation*, en boucle jusqu'à la réponse finale.

Plan (dans l'ordre de l'énoncé) :
1. Créer un simple agent
2. Ajouter une mémoire à l'agent
3. Ajouter des tools à l'agent
4. Utiliser des tools prédéfinis : DuckDuckGo, Tavily, PythonREPLTool
5. Ajouter des middlewares : `dynamic_model`, `dynamic_prompt`, gestion des erreurs des tools, Guardrails, Human In The Loop
6. Créer un chatbot basé sur un RAG agentique

**LLM utilisé :** `qwen3-vl:8b` servi localement par **Ollama** (aucune clé API requise), via `ChatOllama`.
C'est un modèle *thinking* : il raisonne toujours avant de répondre (d'où des temps de réponse de l'ordre de la minute).

In [1]:
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage, ToolMessage
from langchain_ollama import ChatOllama

load_dotenv(override=True)

MODELE = "qwen3-vl:8b"
# reasoning=False : le raisonnement interne du modèle (thinking) n'est pas renvoyé dans la réponse
# num_ctx : fenêtre de contexte d'Ollama (4096 par défaut, trop court pour les résultats de recherche web)
llm = ChatOllama(model=MODELE, temperature=0, reasoning=False, num_ctx=16384)


def trace(resp, debut=0):
    """Affiche le cycle ReAct (à partir du message `debut`) : appels de tools, observations, réponse finale."""
    for m in resp["messages"][debut:]:
        if isinstance(m, AIMessage) and m.tool_calls:
            for tc in m.tool_calls:
                print(f"🛠  Action      : {tc['name']}({tc['args']})")
        elif isinstance(m, ToolMessage):
            print(f"👁  Observation : {str(m.content)[:200]}")
    display(Markdown("**Réponse :** " + resp["messages"][-1].content))

## 1. Simple agent

`create_agent` construit un agent ReAct (un graphe LangGraph prêt à l'emploi) à partir d'un modèle, d'une liste de tools et d'un system prompt.

In [2]:
agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are a helpful assistant",
)

response = agent.invoke({"messages": [{"role": "user", "content": "My name is Mohamed"}]})
display(Markdown(response["messages"][-1].content))

Hello, Mohamed! 😊 It's great to meet you. How can I assist you today? Feel free to ask me anything!

In [3]:
response = agent.invoke({"messages": [{"role": "user", "content": "What is my name?"}]})
display(Markdown(response["messages"][-1].content))

I don't have access to your personal information, so I don't know your name. How can I assist you today? 😊

➡️ Sans mémoire, chaque `invoke` est indépendant : l'agent **ne connaît pas** le nom donné au tour précédent.

### Configuration et contrôle du modèle
On peut instancier directement la classe du fournisseur (`ChatOllama` ici, `ChatOpenAI` dans le cours) pour régler température, longueur maximale, etc. `debug=True` affiche chaque étape du graphe.

In [4]:
llm_config = ChatOllama(model=MODELE, temperature=0.1, num_predict=1000, reasoning=False, num_ctx=16384)

agent = create_agent(model=llm_config, tools=[], system_prompt="You are a helpful assistant", debug=True)
response = agent.invoke({"messages": [HumanMessage("La ville la plus grande au Maroc")]})
display(Markdown(response["messages"][-1].content))

[values] {'messages': [HumanMessage(content='La ville la plus grande au Maroc', additional_kwargs={}, response_metadata={}, id='0de6de12-374b-4b8b-b740-9c26ea7e5efd')]}


[updates] {'model': {'messages': [AIMessage(content="La ville la plus grande au Maroc est **Casablanca**. Elle est située sur la côte atlantique et est le centre économique, commercial et industriel principal du pays. Selon les données démographiques récentes, Casablanca compte environ **3,5 millions d'habitants** (zone urbaine), ce qui la place en tête des villes marocaines en termes de population. \n\nBien que **Rabat** soit la capitale du Maroc, elle est moins peuplée (environ 1,3 million d'habitants). D'autres villes importantes comme Marrakech ou Fès sont également connues pour leur richesse historique et culturelle, mais elles ne dépassent pas Casablanca en taille démographique. \n\nCasablanca est également un important centre portuaire et un hub international, attirant des investissements et des échanges commerciaux significatifs.", additional_kwargs={}, response_metadata={'model': 'qwen3-vl:8b', 'created_at': '2026-11-23T02:59:31.527286Z', 'done': True, 'done_reason': 'stop', '

La ville la plus grande au Maroc est **Casablanca**. Elle est située sur la côte atlantique et est le centre économique, commercial et industriel principal du pays. Selon les données démographiques récentes, Casablanca compte environ **3,5 millions d'habitants** (zone urbaine), ce qui la place en tête des villes marocaines en termes de population. 

Bien que **Rabat** soit la capitale du Maroc, elle est moins peuplée (environ 1,3 million d'habitants). D'autres villes importantes comme Marrakech ou Fès sont également connues pour leur richesse historique et culturelle, mais elles ne dépassent pas Casablanca en taille démographique. 

Casablanca est également un important centre portuaire et un hub international, attirant des investissements et des échanges commerciaux significatifs.

## 2. Mémoire de l'agent

On ajoute un **checkpointer** (`InMemorySaver`) : l'état (l'historique des messages) est sauvegardé par `thread_id`.
Deux appels avec le même `thread_id` partagent la même conversation.

In [5]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant",
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "1"}}
res = agent.invoke({"messages": [HumanMessage("My Name is Mohamed")]}, config=config)
print(res["messages"][-1].content)

Hello, Mohamed! 😊 It's great to meet you. How can I assist you today? Feel free to ask me anything!


In [6]:
res = agent.invoke({"messages": [HumanMessage("What is my name?")]}, config=config)
print(res["messages"][-1].content)

Your name is **Mohamed**! 😊 I remember you mentioned it earlier. How can I assist you today?


In [7]:
# Un autre thread = une autre conversation : l'agent ne connaît pas le nom
res = agent.invoke({"messages": [HumanMessage("What is my name?")]}, config={"configurable": {"thread_id": "2"}})
print(res["messages"][-1].content)

I don't have access to your personal information, so I don't know your name. How can I assist you today? 😊


In [8]:
# Contenu de la mémoire du thread 1
for m in agent.get_state(config).values["messages"]:
    m.pretty_print()

================================ Human Message =================================

My Name is Mohamed
================================== Ai Message ==================================

Hello, Mohamed! 😊 It's great to meet you. How can I assist you today? Feel free to ask me anything!
================================ Human Message =================================

What is my name?
================================== Ai Message ==================================

Your name is **Mohamed**! 😊 I remember you mentioned it earlier. How can I assist you today?
